## Imports


In [ ]:


import subprocess, sys
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('kaggle', 'timm', 'albumentations>=1.3', 'tifffile', 'torchstain')

import os, gc, glob, json, pathlib, zipfile, time, warnings, random
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.backends import cudnn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

cudnn.benchmark = True
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)



## Config

In [ ]:
KAGGLE_TOKEN    = 'placeHolder'
KAGGLE_USERNAME = 'placeHolder'

DATA_DIR    = './data'
MODELS_DIR  = './models_ablation'
os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

BACKBONE    = 'convnext_base'
IMG_SIZE    = 768               # same as main model
BATCH_SIZE  = 2
ACCUM       = 8                 # effective batch = 16, same as main on T4
EPOCHS      = 30
LR          = 2e-4
FOLDS       = 5
TRAIN_FOLDS = [0]
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
SUBMISSION_FILE = 'submission_no_film.csv'

ORGAN_MAP = {'kidney': 0, 'prostate': 1, 'largeintestine': 2, 'spleen': 3, 'lung': 4}

# Per-organ default pixel sizes (µm/pixel) — identical to main pipeline
ORGAN_PIXEL_SIZE_DEFAULTS = {
    'kidney': 0.50, 'prostate': 0.27, 'largeintestine': 0.23,
    'spleen': 0.49, 'lung': 0.50, 'unknown': 0.40,
}
REF_PIXEL_SIZE = 0.4  # µm/pixel — HPA canonical resolution

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


# ==============================================================================
# KAGGLE AUTH
# ==============================================================================
_kdir = pathlib.Path.home() / '.kaggle'
_kdir.mkdir(parents=True, exist_ok=True)
(_kdir / 'access_token').write_text(KAGGLE_TOKEN)
os.chmod(_kdir / 'access_token', 0o600)
print("✓ Kaggle auth configured")


def kaggle_dl(cmd, max_retries=5):
    wait = 60
    for _ in range(max_retries):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            return r
        if '429' in r.stdout + r.stderr:
            print(f"  Rate limited — waiting {wait}s…"); time.sleep(wait); wait = min(wait*2, 300)
        else:
            raise subprocess.CalledProcessError(r.returncode, cmd, r.stdout, r.stderr)
    raise RuntimeError('Max retries exceeded')


def fetch_hpa_images_for_pseudo_labels(pseudo_dir: str, data_dir: str) -> list[str]:
    """Dirs where competition stores HPA TIFFs for pseudo-label IDs (same as ablation_hpc)."""
    test_img_dir  = os.path.join(data_dir, 'test_images')
    train_img_dir = os.path.join(data_dir, 'train_images')
    found_dirs = []
    for d in (test_img_dir, train_img_dir):
        if os.path.isdir(d):
            n = sum(1 for f in os.listdir(d) if f.endswith(('.tiff', '.tif', '.png')))
            print(f"  Found competition images dir: {d}  ({n} images)")
            found_dirs.append(d)
    if not found_dirs:
        print("  [WARN] No competition image directories found — HPA images unavailable.")
    return found_dirs


def fetch_lung_from_pseudo_csv(pseudo_dir: str, dest_dir: str,
                               competition_search_dirs: list | None = None,
                               n: int = 500) -> list:
    """
    Lung manifest from pseudo-label CSVs + Zenodo Team_2 fallback — identical logic to
    ablation_hpc.py / hubmap_hpc.py (HTTP zip to /tmp, extract lung files, copy to dest_dir).
    """
    import tempfile
    import urllib.request

    local_dir = os.path.join(dest_dir, 'lung_hpa')
    os.makedirs(local_dir, exist_ok=True)
    done_f = os.path.join(local_dir, '.done')
    mf_f   = os.path.join(local_dir, 'manifest.json')

    if os.path.exists(done_f) and os.path.exists(mf_f):
        with open(mf_f) as f:
            cached = json.load(f)
        cached = [r for r in cached if os.path.exists(r['img_path'])]
        n_masked = sum(1 for r in cached if r.get('mask_path') or r.get('rle'))

        zenodo_done_check = os.path.join(local_dir, '.zenodo_done')
        lung_csvs_exist   = bool(glob.glob(
            os.path.join(pseudo_dir, '**', '*lung*.csv'), recursive=True))

        if n_masked == 0 and lung_csvs_exist and not os.path.exists(zenodo_done_check):
            print(f"  [CACHE INVALID] Lung cache has 0 masks, Zenodo not yet tried — "
                  f"rebuilding …")
            os.remove(done_f)
            os.remove(mf_f)
        else:
            if n_masked < len(cached) and lung_csvs_exist:
                print(f"  [CACHE PATCH] {len(cached)-n_masked} entries lack RLE — "
                      f"matching against lung CSVs …")
                rle_lookup = {}
                for csv_path in glob.glob(
                        os.path.join(pseudo_dir, '**', '*lung*.csv'), recursive=True):
                    try:
                        _df = pd.read_csv(csv_path)
                        if 'encoding' in _df.columns and 'rle' not in _df.columns:
                            _df = _df.rename(columns={'encoding': 'rle'})
                        if 'id' in _df.columns and 'rle' in _df.columns:
                            for _, _r in _df.iterrows():
                                if pd.notna(_r.get('rle')) and \
                                        str(_r['rle']) not in ('', 'nan'):
                                    rle_lookup[str(_r['id'])] = str(_r['rle'])
                    except Exception:
                        pass
                print(f"    RLE lookup built: {len(rle_lookup)} lung IDs")
                patched = 0
                for r in cached:
                    if r.get('rle') or r.get('mask_path'):
                        continue
                    stem = os.path.splitext(os.path.basename(r['img_path']))[0]
                    rle = rle_lookup.get(stem)
                    if rle is None:
                        suffix = stem.split('_')[-1]
                        for csv_id, csv_rle in rle_lookup.items():
                            if csv_id.endswith('_' + suffix):
                                rle = csv_rle
                                break
                    if rle:
                        r['rle'] = rle
                        patched += 1
                if patched:
                    with open(mf_f, 'w') as fout:
                        json.dump(cached, fout, indent=2)
                    print(f"    Patched {patched} entries with RLE — manifest updated")
                n_masked = sum(1 for r in cached if r.get('mask_path') or r.get('rle'))
            print(f"  [CACHED] Lung — {len(cached)} entries ({n_masked} with masks/RLE)")
            return cached

    lung_csvs = glob.glob(os.path.join(pseudo_dir, '**', '*lung*.csv'), recursive=True)
    print(f"  Found {len(lung_csvs)} lung CSVs: {[os.path.basename(c) for c in lung_csvs]}")

    all_rows = []
    for csv_path in lung_csvs:
        try:
            df = pd.read_csv(csv_path)
            if 'encoding' in df.columns and 'rle' not in df.columns:
                df = df.rename(columns={'encoding': 'rle'})
            if 'id' in df.columns and 'rle' in df.columns:
                df = df[df['rle'].notna() & (df['rle'] != '')].copy()
                df['_source'] = os.path.basename(csv_path)
                all_rows.append(df[['id', 'rle', '_source']])
                print(f"    {os.path.basename(csv_path)}: {len(df)} masked lung rows")
        except Exception as e:
            print(f"    [WARN] Could not read {csv_path}: {e}")

    if not all_rows:
        print("  No lung pseudo-label CSVs found — lung data unavailable.")
        pathlib.Path(done_f).touch()
        with open(mf_f, 'w') as f:
            json.dump([], f)
        return []

    lung_df = pd.concat(all_rows, ignore_index=True).drop_duplicates('id')
    print(f"  Total unique lung IDs with pseudo-labels: {len(lung_df)}")

    if len(lung_df) > n:
        lung_df = lung_df.sample(n=n, random_state=42).reset_index(drop=True)
        print(f"  Sampled down to {n} rows")

    search_dirs = list(competition_search_dirs or [])
    search_dirs.append(local_dir)

    disk_lookup = {}
    for sdir in search_dirs:
        if not os.path.isdir(sdir):
            continue
        for fname in os.listdir(sdir):
            stem = os.path.splitext(fname)[0]
            fpath = os.path.join(sdir, fname)
            if stem not in disk_lookup:
                disk_lookup[stem] = fpath

    manifest   = []
    need_dl    = []

    for _, row in lung_df.iterrows():
        img_id = str(row['id'])
        rle    = row['rle']
        found = disk_lookup.get(img_id)
        if not found:
            for sdir in search_dirs:
                for ext in ('.jpg', '.jpeg', '.png', '.tif', '.tiff'):
                    p = os.path.join(sdir, img_id + ext)
                    if os.path.exists(p):
                        found = p
                        break
                if found:
                    break
        if found:
            manifest.append({
                'img_path':    found,
                'mask_path':   None,
                'rle':         rle,
                'organ':       'lung',
                'data_source': 'HPA',
                'pixel_size':  0.50,
            })
        else:
            need_dl.append((img_id, rle))

    print(f"  Found on disk: {len(manifest)} lung images")
    print(f"  Need download: {len(need_dl)} lung images")

    ZENODO_URL  = 'https://zenodo.org/records/7545745/files/Team_2.zip'
    zenodo_done = os.path.join(local_dir, '.zenodo_done')
    zenodo_dir  = os.path.join(local_dir, 'zenodo_team2')

    if len(manifest) < 50 and not os.path.exists(zenodo_done):
        print(f"\n  ⚠ Only {len(manifest)} lung images found locally — "
              f"trying Zenodo Team_2.zip fallback (3.6 GB, no Kaggle API) …")
        os.makedirs(zenodo_dir, exist_ok=True)
        _tmp_dir = tempfile.mkdtemp(prefix='zenodo_')
        zenodo_zip = os.path.join(_tmp_dir, 'Team_2.zip')
        try:
            print(f"  Downloading {ZENODO_URL} → /tmp/ (not persistent root) …")
            req = urllib.request.urlopen(ZENODO_URL, timeout=60)
            total = int(req.headers.get('Content-Length', 0))
            chunk = 1024 * 1024
            downloaded = 0
            with open(zenodo_zip, 'wb') as fout:
                while True:
                    data = req.read(chunk)
                    if not data:
                        break
                    fout.write(data)
                    downloaded += len(data)
                    if total:
                        pct = downloaded / total * 100
                        print(f"\r  {downloaded/1e6:.0f}/{total/1e6:.0f} MB  ({pct:.1f}%)",
                              end='', flush=True)
            print()
            print("  Extracting lung images from Team_2.zip …")
            lung_found = 0
            _tmp_extract = os.path.join(_tmp_dir, 'extracted')
            os.makedirs(_tmp_extract, exist_ok=True)
            with zipfile.ZipFile(zenodo_zip, 'r') as zf:
                for member in zf.namelist():
                    if 'lung' in member.lower() and not member.endswith('/'):
                        zf.extract(member, _tmp_extract)
                        lung_found += 1
            os.remove(zenodo_zip)
            import shutil as _shu
            for root, _, fnames in os.walk(_tmp_extract):
                for fname in fnames:
                    src = os.path.join(root, fname)
                    rel = os.path.relpath(src, _tmp_extract)
                    dst = os.path.join(zenodo_dir, rel)
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    _shu.copy2(src, dst)
            _shu.rmtree(_tmp_dir, ignore_errors=True)
            pathlib.Path(zenodo_done).touch()
            print(f"  Extracted {lung_found} lung-related files from Team_2.zip")
            rle_lookup = {}
            lung_csvs_loop = glob.glob(
                os.path.join(pseudo_dir, '**', '*lung*.csv'), recursive=True)
            for csv_path in lung_csvs_loop:
                try:
                    _df = pd.read_csv(csv_path)
                    if 'encoding' in _df.columns and 'rle' not in _df.columns:
                        _df = _df.rename(columns={'encoding': 'rle'})
                    if 'id' in _df.columns and 'rle' in _df.columns:
                        for _, _r in _df.iterrows():
                            if pd.notna(_r.get('rle')) and str(_r['rle']) not in ('', 'nan'):
                                rle_lookup[str(_r['id'])] = str(_r['rle'])
                except Exception:
                    pass
            print(f"  Built RLE lookup: {len(rle_lookup)} entries from lung CSVs")

            IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
            n_rle_matched = 0
            for root, _, fnames in os.walk(zenodo_dir):
                for fname in fnames:
                    if os.path.splitext(fname)[1].lower() not in IMAGE_EXTS:
                        continue
                    if 'mask' in fname.lower():
                        continue
                    img_p  = os.path.join(root, fname)
                    stem   = os.path.splitext(fname)[0]
                    rle = rle_lookup.get(stem)
                    if rle is None:
                        suffix = stem.split('_')[-1]
                        for csv_id, csv_rle in rle_lookup.items():
                            if csv_id.endswith('_' + suffix):
                                rle = csv_rle
                                break
                    mask_p = os.path.join(root, stem + '_mask.png')
                    has_mask_file = os.path.exists(mask_p)
                    if rle:
                        n_rle_matched += 1
                    manifest.append({
                        'img_path':    img_p,
                        'mask_path':   mask_p if has_mask_file else None,
                        'rle':         rle,
                        'organ':       'lung',
                        'data_source': 'HPA',
                        'pixel_size':  0.50,
                    })
            print(f"  RLE matched: {n_rle_matched}/{lung_found} Zenodo lung images")
        except Exception as e:
            print(f"  ⚠ Zenodo download failed: {e}")
            print(f"  Continuing with {len(manifest)} lung images found so far.")

    n_masked = sum(1 for r in manifest if r.get('mask_path') or r.get('rle'))
    with open(mf_f, 'w') as f:
        json.dump(manifest, f, indent=2)
    pathlib.Path(done_f).touch()
    print(f"\n  ✓ {len(manifest)} lung entries ready  ({n_masked} with masks/RLE)")
    return manifest


# Competition data
if not os.path.exists(os.path.join(DATA_DIR, 'train.csv')):
    print("Downloading competition data…")
    kaggle_dl(['kaggle', 'competitions', 'download',
               '-c', 'hubmap-organ-segmentation', '-p', DATA_DIR])
    for z in glob.glob(os.path.join(DATA_DIR, '*.zip')):
        with zipfile.ZipFile(z) as zf: zf.extractall(DATA_DIR)
        os.remove(z)
else:
    print("Competition data cached.")

# Pseudo-labels
PSEUDO_DIR   = os.path.join(DATA_DIR, 'pseudo_labels')
_pseudo_done = os.path.join(PSEUDO_DIR, '.done')
if not os.path.exists(_pseudo_done):
    print("Downloading pseudo-labels…")
    os.makedirs(PSEUDO_DIR, exist_ok=True)
    try:
        kaggle_dl(['kaggle', 'datasets', 'download',
                   '-d', 'vladimirsydor/hubmap-2022-add-data-labels-v2',
                   '-p', PSEUDO_DIR])
        for z in glob.glob(os.path.join(PSEUDO_DIR, '*.zip')):
            with zipfile.ZipFile(z) as zf: zf.extractall(PSEUDO_DIR)
            os.remove(z)
        pathlib.Path(_pseudo_done).touch()
    except Exception as e:
        print(f"  ⚠ Pseudo-label download failed: {e}")
else:
    print("Pseudo-labels cached.")

# External lung cache (Zenodo Team_2 — same as ablation_hpc)
EXT_DIR = os.path.join(DATA_DIR, 'external')
os.makedirs(EXT_DIR, exist_ok=True)
HUBMAP_IMG_DIR = os.path.join(DATA_DIR, 'train_images')

print("\n── Locating HPA images (competition test_images + train_images) ──")
COMPETITION_IMG_DIRS = fetch_hpa_images_for_pseudo_labels(PSEUDO_DIR, DATA_DIR)

print("\n── Lung external data (Zenodo fallback — same as ablation_hpc) ──")
LUNG_MANIFEST = fetch_lung_from_pseudo_csv(
    PSEUDO_DIR, EXT_DIR,
    competition_search_dirs=COMPETITION_IMG_DIRS,
    n=500,
)
_n_lung_ok = sum(1 for r in LUNG_MANIFEST if r.get('mask_path') or r.get('rle'))
print(f"  Lung mask/RLE check: {_n_lung_ok}/{len(LUNG_MANIFEST)} images have masks")


## Utils and Dataset

In [ ]:
def rle_decode(rle, shape):
    s = rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[::2], s[1::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape[1], shape[0]).T


def rle_encode(img):
    pixels = img.T.flatten()
    pixels[0] = 0; pixels[-1] = 0
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 2
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


def _load_image_safe(path, max_dim=1024):
    """Load image avoiding RAM spikes from huge TIFFs.

    HuBMAP TIFFs are pyramid files (e.g. 31000×28000 at level 0, ~2000×1800
    at level 2). We pick the coarsest level that still exceeds max_dim so we
    never decompress a 5 GB array just to crop a 512-px patch from it.
    Falls back to cv2 for non-TIFF files or if tifffile fails.
    """
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.tiff', '.tif'):
        try:
            import tifffile
            with tifffile.TiffFile(path) as tf:
                series = tf.series[0]
                chosen = 0
                for lvl, level in enumerate(series.levels):
                    sh = level.shape
                    # shape can be (H,W), (H,W,C) or (C,H,W) etc.
                    h = sh[-2] if len(sh) >= 2 else sh[0]
                    w = sh[-1]
                    if max(h, w) >= max_dim:
                        chosen = lvl   # keep updating — take the last level still ≥ max_dim
                    else:
                        break
                arr = series.levels[chosen].asarray()
                # Normalise axis order to HWC
                if arr.ndim == 3 and arr.shape[0] <= 4:
                    arr = np.transpose(arr, (1, 2, 0))
                return arr
        except Exception:
            pass
    return cv2.imread(path, cv2.IMREAD_UNCHANGED)


def ensure_3ch(img):
    """Return a BGR uint8 HWC image — same channel handling as main pipeline."""
    if img is None: return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        lo, hi = img.min(), img.max()
        if hi > lo: img = (img - lo) / (hi - lo) * 255.0
        img = img.clip(0, 255).astype(np.uint8)
    if img.ndim == 2:
        return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 1:
        return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4:
        return cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return img[:, :, :3]


def preprocess_inputs(x):
    """Identical to main pipeline: scale to [-1, 1]."""
    x = np.asarray(x, dtype='float32')
    x /= 127.0; x -= 1.0
    return x


# Stain normalization 
_STAIN_NORMALIZER = None

def _build_stain_normalizer(hubmap_img_dir: str) -> None:
    global _STAIN_NORMALIZER
    if _STAIN_NORMALIZER is not None:
        return
    imgs = sorted(glob.glob(os.path.join(hubmap_img_dir, '*.tiff')))
    if not imgs:
        return
    ref_bgr = cv2.imread(imgs[0], cv2.IMREAD_COLOR)
    if ref_bgr is None:
        return
    ref_rgb = cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2RGB)
    ref_t   = torch.from_numpy(ref_rgb).permute(2, 0, 1).float()
    try:
        import torchstain
        normalizer = torchstain.normalizers.MacenkoNormalizer(backend='torch')
        normalizer.fit(ref_t)
        _STAIN_NORMALIZER = normalizer
        print(f"  Stain normalizer fitted on {os.path.basename(imgs[0])}")
    except Exception as e:
        print(f"  [WARN] Macenko fit failed ({e}) — stain normalization disabled.")


def _normalize_stain(img_bgr: np.ndarray) -> np.ndarray:
    """Normalize a BGR uint8 image. Falls back to input on any failure."""
    if _STAIN_NORMALIZER is None:
        return img_bgr
    try:
        rgb    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        t      = torch.from_numpy(rgb).permute(2, 0, 1).float()
        result = _STAIN_NORMALIZER.normalize(I=t, stains=True)
        normed = result[0] if isinstance(result, (tuple, list)) else result
        if not isinstance(normed, torch.Tensor) or normed.ndim != 3 or normed.shape[0] != 3:
            return img_bgr
        normed_np = normed.permute(1, 2, 0).clamp(0, 255).byte().numpy()
        if normed_np.shape != img_bgr.shape:
            return img_bgr
        return cv2.cvtColor(normed_np, cv2.COLOR_RGB2BGR)
    except Exception:
        return img_bgr


# Simple augmentation 
_HEAVY_AUG_ORGANS = {'lung', 'spleen'}

def _augment(img, mask, organ):
    if np.random.rand() > 0.5:
        img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
    if np.random.rand() > 0.5:
        img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
    if organ in _HEAVY_AUG_ORGANS:
        k = np.random.choice([0, 1, 2, 3])
        if k:
            img  = np.rot90(img,  k=k).copy()
            mask = np.rot90(mask, k=k).copy()
        alpha = 1.0 + np.random.uniform(-0.15, 0.15)
        beta  = np.random.uniform(-20, 20)
        img   = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        if np.random.rand() > 0.6:
            ksize = np.random.choice([3, 5])
            img   = cv2.GaussianBlur(img, (ksize, ksize), 0)
    return img, mask


# Data manifest

def build_manifest():
    df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
    rows = []
    imgs_dir = os.path.join(DATA_DIR, 'train_images')
    for _, r in df.iterrows():
        p = os.path.join(imgs_dir, f"{r['id']}.tiff")
        if not os.path.exists(p) or pd.isna(r.get('rle', np.nan)): continue
        organ = r['organ']
        rows.append({'img_path': p, 'rle': r['rle'], 'mask_path': None,
                     'organ': organ,
                     'data_source': 'Hubmap',
                     'pixel_size': ORGAN_PIXEL_SIZE_DEFAULTS.get(organ, 0.40)})

    # HPA pseudo-labels
    skip = {'hpa_lungs.csv', 'hpa_spleen.csv'}
    csvs = [c for c in glob.glob(os.path.join(PSEUDO_DIR, '**', '*.csv'), recursive=True)
            if os.path.basename(c) not in skip]

    img_idx = {}
    for root, _, fnames in os.walk(PSEUDO_DIR):
        for fn in fnames:
            if fn.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.tif')):
                img_idx[os.path.splitext(fn)[0].lower()] = os.path.join(root, fn)

    seen = set()
    for csv_p in csvs:
        try:
            pdf = pd.read_csv(csv_p)
            if 'encoding' in pdf.columns and 'rle' not in pdf.columns:
                pdf = pdf.rename(columns={'encoding': 'rle'})
            if 'id' not in pdf.columns or 'rle' not in pdf.columns: continue
            pdf = pdf[pdf['rle'].notna() & (pdf['rle'].astype(str) != 'nan')]
            pdf['_s'] = pdf['id'].astype(str).str.lower()
            pdf = pdf[pdf['_s'].isin(img_idx)]
            for _, r in pdf.iterrows():
                iid = str(r['id'])
                if iid in seen: continue
                org = str(r.get('organ', '')).lower()
                if org not in ORGAN_MAP: continue
                rows.append({'img_path': img_idx[r['_s']], 'rle': str(r['rle']),
                             'mask_path': None,
                             'organ': org, 'data_source': 'HPA',
                             'pixel_size': ORGAN_PIXEL_SIZE_DEFAULTS.get(org, 0.40)})
                seen.add(iid)
        except Exception as e:
            print(f"  [WARN] {os.path.basename(csv_p)}: {e}")

    # Zenodo / pseudo lung manifest (hpa_lungs.csv RLEs + Team_2 images — ablation_hpc parity)
    for r in LUNG_MANIFEST:
        pth = r.get('img_path')
        if not pth or not os.path.exists(pth):
            continue
        rows.append({
            'img_path':    pth,
            'rle':         str(r.get('rle') or ''),
            'mask_path':   r.get('mask_path'),
            'organ':       'lung',
            'data_source': r.get('data_source', 'HPA'),
            'pixel_size':  float(r.get('pixel_size', 0.50)),
        })

    print(f"Manifest: {len(rows)} rows  "
          f"({sum(1 for r in rows if r['data_source']=='Hubmap')} HuBMAP  "
          f"{sum(1 for r in rows if r['data_source']=='HPA')} HPA "
          f"— includes {len(LUNG_MANIFEST)} lung-manifest entries)")
    return pd.DataFrame(rows)


print("\nBuilding manifest…")
MANIFEST = build_manifest()

# Fit stain normalizer on first available HuBMAP training TIFF
_build_stain_normalizer(os.path.join(DATA_DIR, 'train_images'))


print("\nBuilding manifest…")
MANIFEST = build_manifest()

# Fit stain normalizer on first available HuBMAP training TIFF
_build_stain_normalizer(os.path.join(DATA_DIR, 'train_images'))

# Dataset

class HubMapDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df       = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = _load_image_safe(row['img_path'], max_dim=IMG_SIZE * 2)
        img  = ensure_3ch(img)
        rle  = str(row.get('rle', ''))
        mp   = row.get('mask_path')
        if mp is not None and pd.notna(mp) and os.path.exists(str(mp)):
            raw = cv2.imread(str(mp), cv2.IMREAD_UNCHANGED)
            if raw is None:
                mask = np.zeros(img.shape[:2], dtype=np.uint8)
            else:
                if raw.ndim != 2:
                    raw = cv2.cvtColor(raw, cv2.COLOR_BGR2GRAY)
                mask = (raw > 127).astype(np.uint8)
        elif rle and rle not in ('nan', ''):
            mask = rle_decode(rle, img.shape[:2])
        else:
            mask = np.zeros(img.shape[:2], dtype=np.uint8)

        # Pre-cap to 2× target (same as main pipeline)
        MAX_PRE = IMG_SIZE * 2
        if img.shape[0] > MAX_PRE or img.shape[1] > MAX_PRE:
            img  = cv2.resize(img,  (MAX_PRE, MAX_PRE))
            mask = cv2.resize(mask, (MAX_PRE, MAX_PRE),
                              interpolation=cv2.INTER_NEAREST)

        # Pixel-scale normalisation (identical to main pipeline)
        pixel_size = float(row.get('pixel_size', REF_PIXEL_SIZE))
        if pixel_size > 0:
            scale = np.clip(pixel_size / REF_PIXEL_SIZE, 0.25, 4.0)
            if abs(scale - 1.0) > 0.05:
                new_h = max(64, int(img.shape[0] * scale))
                new_w = max(64, int(img.shape[1] * scale))
                img  = cv2.resize(img,  (new_w, new_h))
                mask = cv2.resize(mask, (new_w, new_h),
                                  interpolation=cv2.INTER_NEAREST)

        # Hard resize to training resolution (same as main pipeline)
        img  = cv2.resize(img,  (IMG_SIZE, IMG_SIZE))
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE),
                          interpolation=cv2.INTER_NEAREST)

        # Stain normalisation — HuBMAP images only (same as main pipeline)
        if str(row.get('data_source', 'HPA')) == 'Hubmap':
            img = _normalize_stain(img)

        # Augmentation (same simple flip strategy as main pipeline)
        if self.is_train:
            img, mask = _augment(img, mask, row['organ'])

        # Tensor conversion (same preprocess_inputs as main pipeline)
        img_t  = torch.from_numpy(
            preprocess_inputs(img).transpose((2, 0, 1)).copy()).float()
        mask_t = torch.from_numpy(mask.copy()).float().unsqueeze(0)
        return {'img': img_t, 'mask': mask_t, 'organ': row[






## Model and Loss Func

In [ ]:
class ConvBnSiLU(nn.Module):
    """Standard conv block — the FiLM-free counterpart of ConvSiluFiLM."""
    def __init__(self, in_ch, out_ch, ks=3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, ks, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class PlainPathologyUNet(nn.Module):
    """
    ConvNext-Large encoder + plain U-Net decoder.
    No metadata input. No FiLM. Purely image-driven predictions.

    Structurally identical to MultimodalPathologyUNet in hubmap_pipeline.py
    except every ConvSiluFiLM is replaced by ConvBnSiLU and all conditioning
    logic is removed. This is the ablation baseline for the paper.
    """
    def __init__(self, encoder_name='convnext_large', pretrained=True):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=pretrained,
                                         features_only=True)
        ef = [f['num_chs'] for f in self.encoder.feature_info]
        self.num_stages = len(ef)
        df = [32, 48, 64, 96, 128]

        self.conv6   = ConvBnSiLU(ef[-1],          df[-1])
        self.conv6_2 = ConvBnSiLU(df[-1]+ef[-2],   df[-1])
        self.conv7   = ConvBnSiLU(df[-1],           df[-2])
        self.conv7_2 = ConvBnSiLU(df[-2]+ef[-3],   df[-2])
        self.conv8   = ConvBnSiLU(df[-2],           df[-3])
        self.conv8_2 = ConvBnSiLU(df[-3]+ef[-4],   df[-3])
        self.conv9   = ConvBnSiLU(df[-3],           df[-4])
        self.conv9_2 = None if self.num_stages == 4 else \
                       ConvBnSiLU(df[-4]+ef[-5], df[-4])
        self.conv10  = ConvBnSiLU(df[-4],           df[-5])
        self.head    = nn.Conv2d(df[-5], 1, 1)

    def forward(self, x):
        feats = self.encoder(x)
        if self.num_stages == 4:
            e2, e3, e4, e5 = feats
        else:
            e1, e2, e3, e4, e5 = feats
        up = lambda t: F.interpolate(t, scale_factor=2, mode='bilinear', align_corners=False)
        d6  = self.conv6(up(e5))
        d6  = self.conv6_2(torch.cat([d6, e4], 1))
        d7  = self.conv7(up(d6))
        d7  = self.conv7_2(torch.cat([d7, e3], 1))
        d8  = self.conv8(up(d7))
        d8  = self.conv8_2(torch.cat([d8, e2], 1))
        d9  = self.conv9(up(d8))
        if self.num_stages == 5:
            d9 = self.conv9_2(torch.cat([d9, e1], 1))
        out = self.head(self.conv10(d9))
        return F.interpolate(out, scale_factor=2, mode='bilinear', align_corners=False)


def build_model(pretrained=True):
    m = PlainPathologyUNet('convnext_base', pretrained=pretrained)
    n = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"  Built PlainPathologyUNet (ConvNext-Large, NO FiLM)  params={n:.1f}M")
    return m

# Loss

class FocalDiceLoss(nn.Module):
    """Identical to main pipeline's FocalDiceLoss."""
    def __init__(self, alpha=0.25, gamma=2.0, dice_weight=0.5, smooth=1e-5):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
        self.dice_weight = dice_weight; self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):
        bce  = self.bce(logits, targets)
        prob = torch.sigmoid(logits)
        pt   = prob * targets + (1 - prob) * (1 - targets)
        fw   = (1 - pt) ** self.gamma
        aw   = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        fl   = (aw * fw * bce).mean()
        i    = (prob * targets).sum(dim=(2, 3))
        u    = prob.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dl   = 1.0 - ((2. * i + self.smooth) / (u + self.smooth)).mean()
        return (1 - self.dice_weight) * fl + self.dice_weight * dl


def dice_score(pred, target, threshold=0.5):
    pred  = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return ((2.*inter + 1e-5) / (union + 1e-5)).mean().item()


## Training

In [ ]:

def train_fold(fold, train_df, val_df):
    print(f"\n{'='*60}")
    print(f"  Fold {fold}  |  train={len(train_df)}  val={len(val_df)}")
    print(f"  ABLATION: ConvNext-Base U-Net — NO FiLM")
    print(f"{'='*60}")

    train_ds = HubMapDataset(train_df, is_train=True)
    val_ds   = HubMapDataset(val_df,   is_train=False)

    w = torch.DoubleTensor([
        {'lung': 3.0, 'spleen': 2.0, 'prostate': 1.5,
         'kidney': 1.0, 'largeintestine': 1.0}.get(r['organ'], 1.0)
        for _, r in train_df.iterrows()
    ])
    sampler = WeightedRandomSampler(w, len(w), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=0, pin_memory=False, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                              num_workers=0, pin_memory=False)

    model     = build_model(pretrained=True).to(DEVICE)
    criterion = FocalDiceLoss().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    # Step-based cosine schedule — identical to main pipeline
    total_steps = len(train_loader) // ACCUM * EPOCHS
    scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda')

    best_val_loss = float('inf')
    best_dice     = 0.0
    no_improve    = 0
    ckpt_path     = os.path.join(MODELS_DIR, f'ablation_fold{fold}_best.pth')

    for epoch in range(EPOCHS):
        #  Train
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1:>3}/{EPOCHS} [train]",
                    dynamic_ncols=True)
        for step, batch in enumerate(pbar):
            imgs  = batch['img'].to(DEVICE)
            masks = batch['mask'].to(DEVICE)
            with torch.amp.autocast('cuda'):
                pred = model(imgs)           # no metadata — ablation
                loss = criterion(pred, masks) / ACCUM
            scaler.scale(loss).backward()
            if (step+1) % ACCUM == 0 or (step+1) == len(train_loader):
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
                scheduler.step()             # step-based, same as main pipeline
            train_loss += loss.item() * ACCUM
            pbar.set_postfix(loss=f"{train_loss/(step+1):.4f}")

        #  Validate
        model.eval()
        val_loss   = 0.0
        organ_dice = {o: [] for o in ORGAN_MAP}
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Ep {epoch+1:>3}/{EPOCHS} [val]  ",
                              dynamic_ncols=True):
                imgs  = batch['img'].to(DEVICE)
                masks = batch['mask'].to(DEVICE)
                organ = batch['organ'][0]
                with torch.amp.autocast('cuda'):
                    pred = model(imgs)
                    val_loss += criterion(pred, masks).item()
                organ_dice[organ].append(dice_score(pred, masks))

        avg_val  = val_loss / len(val_loader)
        per_organ = {o: float(np.mean(s)) for o, s in organ_dice.items() if s}
        avg_dice  = float(np.mean(list(per_organ.values())))

        organ_str = '  '.join(f"{k[:3]}={v:.3f}" for k, v in per_organ.items())
        print(f"\n  Ep {epoch+1:>3}  train={train_loss/len(train_loader):.4f}  "
              f"val={avg_val:.4f}  dice={avg_dice:.4f}  "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")
        print(f"  {organ_str}")

        # Save on best val loss 
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_dice     = avg_dice
            no_improve    = 0
            torch.save({'epoch': epoch+1, 'state_dict': model.state_dict(),
                        'val_loss': avg_val, 'dice': avg_dice,
                        'per_organ': per_organ}, ckpt_path)
            if _DRIVE_CKPT:
                import shutil as _sh
                _sh.copy2(ckpt_path, _DRIVE_CKPT)
                print(f"  ✓ NEW BEST  val={best_val_loss:.4f}  dice={best_dice:.4f}"
                      f"  → saved locally + Drive")
            else:
                print(f"  ✓ NEW BEST  val={best_val_loss:.4f}  dice={best_dice:.4f}"
                      f"  → saved locally")
        else:
            no_improve += 1
            if no_improve >= 10:
                print(f"\n  Early stopping at epoch {epoch+1}")
                break

    print(f"\n  Fold {fold} best val_loss={best_val_loss:.4f}  dice={best_dice:.4f}")
    return ckpt_path, best_dice

# Cross validation

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
hubmap_df = MANIFEST[MANIFEST['data_source'] == 'Hubmap'].reset_index(drop=True)
ext_df    = MANIFEST[MANIFEST['data_source'] != 'Hubmap'].reset_index(drop=True)
splits    = list(skf.split(hubmap_df, hubmap_df['organ']))

best_ckpts = []
for fold in TRAIN_FOLDS:
    tr_idx, va_idx = splits[fold]
    train_df = pd.concat([hubmap_df.iloc[tr_idx], ext_df], ignore_index=True)
    val_df   = hubmap_df.iloc[va_idx].reset_index(drop=True)
    ckpt, dice = train_fold(fold, train_df, val_df)
    best_ckpts.append((ckpt, dice))


## Inference

In [ ]:
# Inference

def run_inference(ckpt_paths):
    print(f"\nRunning inference (no-FiLM model)…")
    models = []
    for ckpt_path, _ in ckpt_paths:
        m = build_model(pretrained=False).to(DEVICE)
        sd = torch.load(ckpt_path, map_location=DEVICE)
        m.load_state_dict(sd['state_dict'])
        m.eval()
        models.append(m)

    test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
    test_dir = os.path.join(DATA_DIR, 'test_images')

    THRESH = {'kidney': 0.40, 'prostate': 0.40, 'largeintestine': 0.35,
              'spleen': 0.40, 'lung': 0.30}

    results = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        img0 = _load_image_safe(os.path.join(test_dir, f"{row['id']}.tiff"),
                                max_dim=IMG_SIZE * 2)
        if img0 is None:
            results.append({'id': row['id'], 'rle': ''}); continue
        img0 = ensure_3ch(img0)
        oh, ow = img0.shape[:2]
        acc = np.zeros((oh, ow), dtype='float32')
        n = 0
        for model in models:
            for flip in [False, True]:
                for k in range(4):
                    aug = img0.copy()
                    if flip: aug = aug[:, ::-1]
                    if k:    aug = np.rot90(aug, k)
                    inp_arr = cv2.resize(aug, (IMG_SIZE, IMG_SIZE))
                    inp = torch.from_numpy(
                        preprocess_inputs(inp_arr).transpose((2, 0, 1)).copy()
                    ).float().unsqueeze(0).to(DEVICE)
                    with torch.no_grad(), torch.amp.autocast('cuda'):
                        logit = model(inp)   # no metadata input
                    p = torch.sigmoid(logit)[0, 0].float().cpu().numpy()
                    if k:    p = np.rot90(p, 4-k)
                    if flip: p = p[:, ::-1]
                    acc += cv2.resize(p, (ow, oh))
                    n += 1
        acc /= n
        mask = (acc > THRESH.get(row['organ'], 0.4)).astype(np.uint8)
        results.append({'id': row['id'], 'rle': rle_encode(mask)})

    pd.DataFrame(results).to_csv(SUBMISSION_FILE, index=False)
    print(f"\n✓ Saved {SUBMISSION_FILE}  ({len(results)} rows)")
    print("\n── ABLATION RESULTS ─────────────────────────────────────")
    for ckpt, dice in ckpt_paths:
        print(f"  {os.path.basename(ckpt)}  val Dice = {dice:.4f}")
    print("  Compare against main model (FiLM) val Dice to measure")
    print("  the contribution of metadata conditioning.")
    print("─────────────────────────────────────────────────────────")


run_inference(best_ckpts)
print("\nDone! Submit submission_no_film.csv to Kaggle and compare with main model.")
